In [ ]:
import google.generativeai as genai
import pandas as pd
import time
import os
from tqdm import tqdm

In [ ]:
# KONFIGURASI GEMINI API & LOAD DATA (Kaggle-ready)

# Ambil API key dari Kaggle Secrets (Recommended) atau env var
api_key = os.environ.get("KAGGLE_GEMINI_API_KEY")
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key = UserSecretsClient().get_secret("KAGGLE_GEMINI_API_KEY")
    except Exception:
        api_key = None

if not api_key:
    raise ValueError("API key tidak ditemukan. Set Kaggle Secret 'KAGGLE_GEMINI_API_KEY'.")

genai.configure(api_key=api_key)

# Konfigurasi Model. Gemini 1.5 Flash sangat cepat dan handal untuk Few-Shot.
model = genai.GenerativeModel('gemini-2.5-flash')

# Ganti path ini dengan dataset Kaggle Anda
default_kaggle_path = "/kaggle/input/your-dataset/datafix.csv"
data_path = default_kaggle_path if os.path.exists(default_kaggle_path) else "datafix.csv"

# Asumsikan 'df' adalah DataFrame utuh Anda SEBELUM PREPROCESSING
# Harus ada kolom: 'IDPSJ', 'IDJwb', 'grade', 'original_text' (teks asli siswa)
# Dan juga asumsi Anda punya dataframe referensi Pertanyaan & Kunci
# df = pd.read_csv('asag_dataset_raw.csv') 
df = pd.read_csv(data_path)

# Standarisasi kolom teks jawaban agar downstream selalu pakai 'original_text'
text_candidates = ['original_text', 'answer', 'answers', 'jawaban', 'text']
text_col = next((c for c in text_candidates if c in df.columns), None)
if text_col is None:
    raise KeyError(f"Kolom teks jawaban tidak ditemukan. Kandidat: {text_candidates}")
df['original_text'] = df[text_col].astype(str).str.strip()

# Pastikan grade numerik integer 1-10
df['grade'] = pd.to_numeric(df['grade'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
df = df.dropna(subset=['grade', 'IDPSJ'])
df['grade'] = df['grade'].round().astype(int).clip(lower=1, upper=10)
df['IDPSJ'] = pd.to_numeric(df['IDPSJ'], errors='coerce')
df = df.dropna(subset=['IDPSJ'])
df['IDPSJ'] = df['IDPSJ'].astype(int)

required_cols = ['IDJwb', 'IDPSJ', 'questions', 'answerKeys', 'grade', 'original_text']
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise KeyError(f"Kolom wajib tidak ada: {missing_cols}")

# Bangun tabel referensi pertanyaan & kunci per IDPSJ
# format: {IDPSJ: {'q': 'teks_tanya', 'ak': 'teks_kunci'}}
soal_kunci_map = (
    df[['IDPSJ', 'questions', 'answerKeys']]
    .drop_duplicates(subset=['IDPSJ'])
    .set_index('IDPSJ')
    .apply(lambda row: {'q': row['questions'], 'ak': row['answerKeys']}, axis=1)
    .to_dict()
)
soal_kunci_map = {int(k): v for k, v in soal_kunci_map.items()}

output_dir = "hasil_sintesis"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Proses kolom grade: konversi dari format "10,00" → integer 1-10
# Ganti koma desimal (format Indonesia) dengan titik, lalu konversi ke float → int
df['grade'] = df['grade'].astype(str).str.replace(',', '.').astype(float).round().astype(int)

# Grade 0 diubah menjadi 1 (nilai minimum valid)
df['grade'] = df['grade'].clip(lower=1)

print("Distribusi grade setelah konversi:")
grade_dist = df['grade'].value_counts().sort_index()
print(grade_dist)

In [ ]:
# FUNGSI INTI SINTESIS DATA (Few-Shot API Call)
def synthesize_data_fewshot(question, answer_key, references_df, target_grade):
    examples_str = ""
    for _, row in references_df.iterrows():
        examples_str += f"- Jawaban Siswa (Grade {row['grade']}): \"{row['original_text']}\"\n"
    
    prompt = f"""
    Anda adalah asisten ahli dalam penilaian esai otomatis (ASAG).
    Tugas Anda adalah membuat contoh jawaban siswa sintetis yang logis, realistis, 
    dan secara akurat mencerminkan kriteria penilaian untuk grade yang diminta.
    
    Konteks Soal:
    Pertanyaan: "{question}"
    Kunci Jawaban Sempurna (Grade 10): "{answer_key}"
    
    Referensi Jawaban Asli:
    {examples_str}
    
    PERMINTAAN SINTESIS:
    Tolong buat SATU contoh jawaban sintetis BARU yang kualitasnya pantas mendapatkan nilai TEPAT {target_grade} dari 10.
    Jika ada referensi Grade {target_grade} di atas, buatlah jawaban yang sepadan kualitasnya dengan referensi tersebut, 
    namun gunakan pilihan kata, struktur kalimat, atau kesalahan yang BERBEDA agar data lebih bervariasi.
    
    ATURAN KETAT OUTPUT:
    1. Berikan LANGSUNG teks jawabannya saja.
    2. Jangan gunakan kata pengantar atau penutup apapun.
    """
    try:
        response = model.generate_content(prompt)
        return response.text.strip().replace('\"', '')
    except Exception as e:
        print(f"Error API: {e}")
        return None

In [ ]:
# MODE MANUAL PER IDPSJ
MANUAL_IDPSJ_LIST = [1]  # Contoh: [1] atau [1, 2, 9]. Isi [] untuk semua IDPSJ

# AUTOMATION LOOP PER IDPSJ (DETEKSI & FILL)
synthetic_rows = []

all_idpsj = sorted(df['IDPSJ'].dropna().astype(int).unique().tolist())
if MANUAL_IDPSJ_LIST:
    manual_clean = sorted(set(int(x) for x in MANUAL_IDPSJ_LIST))
    idpsj_list = [x for x in manual_clean if x in all_idpsj]
    missing_manual = [x for x in manual_clean if x not in all_idpsj]
    if missing_manual:
        print(f"IDPSJ ini tidak ditemukan di data dan akan dilewati: {missing_manual}")
else:
    idpsj_list = all_idpsj

TARGET_MIN_DATA = 5  # Target minimal jumlah jawaban per grade (1-10)
MAX_REFERENCES = 5    # Maksimal contoh jawaban yang disuapkan ke prompt

print(f"Memulai sintesis untuk IDPSJ: {idpsj_list}")
print(f"Target: Minimal {TARGET_MIN_DATA} data per grade (1-10) untuk setiap IDPSJ terpilih.")

for idpsj in tqdm(idpsj_list, desc="Processing IDPSJ"):
    current_df = df[(df['IDPSJ'] == idpsj) & (~df['IDJwb'].astype(str).str.startswith('syn_'))].copy()

    if current_df.empty:
        continue

    if idpsj not in soal_kunci_map:
        continue

    question_txt = soal_kunci_map[idpsj]['q']
    key_txt = soal_kunci_map[idpsj]['ak']

    # Loop untuk setiap kemungkinan nilai 1 sampai 10
    for target_g in range(1, 11):
        # Hitung berapa banyak jawaban asli yang sudah ada untuk nilai ini
        existing_answers = current_df[current_df['grade'] == target_g]
        current_count = len(existing_answers)

        # Hitung berapa yang perlu disintesis
        needed = TARGET_MIN_DATA - current_count

        if needed > 0:
            # --- LOGIKA PENCARIAN REFERENSI (Tetangga Terdekat) ---
            refs_list = []

            # 1. Masukkan jawaban yang sudah ada di target_grade (jika ada)
            if current_count > 0:
                # Ambil sampel (maksimal sesuai MAX_REFERENCES)
                sample_existing = existing_answers.sample(min(current_count, MAX_REFERENCES))
                refs_list.append(sample_existing)

            current_ref_count = sum([len(r) for r in refs_list])

            # 2. Jika referensi masih kurang dari MAX_REFERENCES, cari ke tetangga terdekat
            if current_ref_count < MAX_REFERENCES:
                available_grades = set(current_df['grade'].unique()) - {target_g}
                # Urutkan grade lain berdasarkan jarak terdekat ke target_g
                sorted_neighbors = sorted(list(available_grades), key=lambda x: abs(x - target_g))

                for neighbor_g in sorted_neighbors:
                    neighbor_data = current_df[current_df['grade'] == neighbor_g]
                    if neighbor_data.empty:
                        continue

                    # Ambil 1 contoh dari tetangga ini untuk memperkaya variasi
                    refs_list.append(neighbor_data.sample(1))

                    current_ref_count += 1
                    if current_ref_count >= MAX_REFERENCES:
                        break

            if not refs_list:
                continue

            # Gabungkan semua referensi terpilih menjadi satu DataFrame
            references_df = pd.concat(refs_list).head(MAX_REFERENCES)

            # --- EKSEKUSI SINTESIS ---
            # Loop sebanyak kekurangan data (needed)
            for i in range(needed):
                syn_text = synthesize_data_fewshot(question_txt, key_txt, references_df, target_g)

                if syn_text:
                    synthetic_rows.append({
                        'IDPSJ': idpsj,
                        'IDJwb': f'syn_{idpsj}_{target_g}_{current_count + i + 1}',
                        'grade': target_g,
                        'original_text': syn_text
                    })
                time.sleep(1.5)  # Jeda API

In [ ]:
# TAMPILKAN HASIL JAWABAN SINTESIS SAJA (tanpa simpan file)
if synthetic_rows:
    df_synthetic = pd.DataFrame(synthetic_rows)
    df_synthetic = df_synthetic.sort_values(['IDPSJ', 'grade', 'IDJwb']).reset_index(drop=True)

    print(f"\nSelesai! Berhasil menyintesis {len(synthetic_rows)} data baru.")

    print("\nSemua nilai (grade) beserta IDPSJ:")
    print(df_synthetic[['IDPSJ', 'grade']].to_string(index=False))

    print("\nDetail semua data sintesis:")
    print(df_synthetic.to_string(index=False))

    # Jika Anda ingin format list of dict untuk input manual
    synthetic_list = df_synthetic.to_dict('records')
    print("\nFormat list (semua data):")
    print(synthetic_list)
else:
    print("\nTidak ada data sintesis yang dihasilkan.")

In [ ]:
# SIMPAN HASIL SINTESIS KE XLSX (kolom: idpsj, answer, grade)
from pathlib import Path
from datetime import datetime

if synthetic_rows:
    # Pastikan DataFrame sintesis tersedia
    if 'df_synthetic' not in globals():
        df_synthetic = pd.DataFrame(synthetic_rows)
        df_synthetic = df_synthetic.sort_values(['IDPSJ', 'grade', 'IDJwb']).reset_index(drop=True)

    # Siapkan kolom output
    df_download = df_synthetic[['IDPSJ', 'original_text', 'grade']].copy()
    df_download = df_download.rename(columns={'IDPSJ': 'idpsj', 'original_text': 'answer', 'grade': 'grade'})

    save_dir = Path(output_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_xlsx = save_dir / f'synthetic_results_{timestamp}.xlsx'

    df_download.to_excel(out_xlsx, index=False)
    print(f'File berhasil disimpan: {out_xlsx.resolve()}')
    print('Silakan buka file tersebut secara manual dari folder hasil_sintesis.')
else:
    print('Belum ada data sintesis untuk disimpan.')